# TP 1: LDA/QDA y optimización matemática de modelos

Se puede consultar la introducción teórica en el mono-notebook, se prefiere mantener este lo más chico posible.

In [2]:
# imports
import numpy as np
import numpy.linalg as LA

from base.qda import QDA, TensorizedQDA
from base.cholesky import QDA_Chol1, QDA_Chol2, QDA_Chol3
from utils.bench import Benchmark
from utils.datasets import (get_iris_dataset, get_letters_dataset, 
                            get_penguins_dataset, get_wine_dataset,
                            label_encode)


## Ejemplo

In [3]:
# levantamos el dataset Wine, que tiene 13 features y 178 observaciones en total
X_full, y_full = get_wine_dataset()

X_full.shape, y_full.shape

((178, 13), (178, 1))

In [4]:
# encodeamos a número las clases
y_full_encoded = label_encode(y_full)

y_full[:5], y_full_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [5]:
# generamos el benchmark
# observar que son valores muy bajos de runs para que corra rápido ahora
b = Benchmark(
    X_full, y_full_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [6]:
# bencheamos un par
to_bench = [QDA]

for model in to_bench:
    b.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [7]:
# como es una clase, podemos seguir bencheando más después
b.bench(TensorizedQDA)

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [8]:
# hacemos un summary
b.summary()

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb
model,,,,,,,,,
QDA,0.132753,0.148889,1.055385,0.679894,0.982407,0.018471,0.000622,0.007697,0.001796
TensorizedQDA,0.121333,0.152809,0.442436,0.352283,0.982593,0.018471,0.000653,0.012123,0.000153


In [9]:
# son muchos datos! nos quedamos con un par nomás
summ = b.summary()

# como es un pandas DataFrame, subseteamos columnas fácil
summ[['train_median_ms', 'test_median_ms','mean_accuracy']]

,train_median_ms,test_median_ms,mean_accuracy
model,,,
QDA,0.132753,1.055385,0.982407
TensorizedQDA,0.121333,0.442436,0.982593


In [10]:
# podemos setear un baseline para que fabrique columnas de comparación
summ = b.summary(baseline='QDA')

summ

,train_median_ms,train_std_ms,test_median_ms,test_std_ms,mean_accuracy,train_mem_median_mb,train_mem_std_mb,test_mem_median_mb,test_mem_std_mb,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,,,,,,,
QDA,0.132753,0.148889,1.055385,0.679894,0.982407,0.018471,0.000622,0.007697,0.001796,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.121333,0.152809,0.442436,0.352283,0.982593,0.018471,0.000653,0.012123,0.000153,1.094121,2.385397,1.0,0.634912


In [11]:
# volvemos a subsetear columnas
summ[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.132753,1.055385,0.982407,1.000000,1.000000,1.0,1.000000
TensorizedQDA,0.121333,0.442436,0.982593,1.094121,2.385397,1.0,0.634912


In [12]:
# Levantamos los datasets Iris, Letters, Penguins y Wine
X_full_iris, y_full_iris = get_iris_dataset()
# X_full_letters, y_full_letters = get_letters_dataset()
# X_full_penguins, y_full_penguins = get_penguins_dataset()
X_full_wine, y_full_wine = get_wine_dataset()

X_full_iris.shape, y_full_iris.shape
# X_full_letters.shape, y_full_letters.shape
# X_full_penguins.shape, y_full_penguins.shape
X_full_wine.shape, y_full_wine.shape

((178, 13), (178, 1))

In [13]:
# Encodeamos a número las clases
y_full_iris_encoded = label_encode(y_full_iris)
y_full_wine_encoded = label_encode(y_full_wine)

y_full_iris[:5], y_full_iris_encoded[:5]
y_full_wine[:5], y_full_wine_encoded[:5]

(array([['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0'],
        ['class_0']], dtype='<U7'),
 array([[0],
        [0],
        [0],
        [0],
        [0]]))

In [14]:
benchmark_iris = Benchmark(
    X_full_iris, y_full_iris_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

benchmark_wine = Benchmark(
    X_full_wine, y_full_wine_encoded,
    n_runs = 100,
    warmup = 20,
    mem_runs = 20,
    test_sz = 0.3,
    same_splits = False
)

Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 105
Test size rows (approx): 45
Test size fraction: 0.3
Benching params:
Total runs: 140
Warmup runs: 20
Peak Memory usage runs: 20
Running time runs: 100
Train size rows (approx): 125
Test size rows (approx): 53
Test size fraction: 0.3


In [15]:
# Benchmark de los modemos en el dataset Iris
to_bench = [QDA, TensorizedQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    benchmark_iris.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [16]:
for model in to_bench:
    benchmark_wine.bench(model)

QDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol1 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol1 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol2 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/20 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
summary_iris = benchmark_iris.summary(baseline='QDA')
summary_iris[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.103997,0.724132,0.970667,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.085742,0.269126,0.971111,1.212906,2.690680,1.012482,0.878836
QDA_Chol1,0.126403,0.463621,0.972889,0.822742,1.561905,0.965481,0.947228
QDA_Chol2,0.118933,1.146992,0.972222,0.874417,0.631331,1.003712,0.934053
QDA_Chol3,0.109343,0.470986,0.977333,0.951108,1.537481,1.002472,1.020702


In [18]:
summary_wine = benchmark_wine.summary(baseline='QDA')
summary_wine[[
    'train_median_ms', 'test_median_ms','mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,0.132268,1.042754,0.982407,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,0.123787,0.474321,0.982593,1.068509,2.198414,1.000000,0.632599
QDA_Chol1,0.150168,0.596504,0.985741,0.880797,1.748109,1.022089,0.976240
QDA_Chol2,0.127338,1.396773,0.983333,1.038716,0.746545,1.029487,0.951586
QDA_Chol3,0.130093,0.602644,0.986111,1.016715,1.730298,1.032134,0.994807


# Consigna QDA

## Tensorización

### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?

> **R:** Paraleliza únicamente sobre las $k$ clases, ya que la cadena de llamadas es: `predict(X) -> predict_one(X) -> predict_log_conditionals(X)` donde X($p$, 1) es una sola observación. La iteración sobre las $n$ observaciones a predecir se sigue haciendo en `fiuba-ceia-amia-tp/base/bayesian.py` en la línea 34.

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.
> **R:** La forma de `tensor_inv_covs` es ($k$, $p$, $p$) y la forma de `tensor_means` es ($k$, $p$, 1). Para predecir, `TensorizedQDA` hace un producto punto entre `tensor_inv_covs` y `tensor_means`. La razón principal por la que predicen lo mismo está en observar el código de `_predict_log_conditional` y `_predict_log_conditionals`: ambas devuelven el cálculo de: $$f_j(x) = \frac{1}{(2 \pi)^\frac{p}{2} \cdot |\Sigma_j|^\frac{1}{2}} e^{- \frac{1}{2}(x-\mu_j)^T \Sigma_j^{-1} (x- \mu_j)}$$ La única diferencia es que `QDA` lo hace para una clase a la vez, y `TensorizedQDA` lo hace para todas las clases en paralelo. En esta misma línea, en ambas clases el método `_predict_one` devuelve el índice de la clase que maximiza la probabilidad condicional a posteriori: $$log P(G=k) + log P(x|G=k)$$ Por lo tanto, ambas clases predicen lo mismo, la diferencia es únicamente en performance, siendo `TensorizedQDA` más rápido al paralelizar sobre las clases.

In [22]:
from base.qda import TensorizedQDA
from utils.datasets import get_wine_dataset, label_encode
from sklearn.model_selection import train_test_split

X, y = get_wine_dataset()
y = label_encode(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# transponer porque la clase espera (n_feat, n_obs)
X_train = X_train.T
y_train = y_train.T

model = TensorizedQDA()
model.fit(X_train, y_train)

print('tensor_inv_cov:', model.tensor_inv_cov.shape)
print('tensor_means:  ', model.tensor_means.shape)


tensor_inv_cov: (3, 13, 13)
tensor_means:   (3, 13, 1)


### 2) Optimización

3. Implementar el modelo `FasterQDA` (se recomienda heredarlo de `TensorizedQDA`) de manera de eliminar el ciclo for en el método predict.
> R: El código implementado se encuentra en [qda.py](base/qda.py#L47-L58).

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.
> **R:** La matriz de $n \times n$ aparece en [qda.py](base/qda.py#L51), donde se calcula `inner_prod`. En esta línea, `unbiased_X` tiene forma ($k$, $p$, $n$) y `tensor_inv_cov` tiene forma ($k$, $p$, $p$). Al hacer el producto punto entre estas dos matrices, se obtiene una matriz de forma ($k$, $n$, $n$).

5. Demostrar que $$diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente $$np.sum(A^T \odot B, axis=0).T$$ queda a preferencia del alumno cuál usar.